In [1]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_validate
from pathlib import Path
import sys
base_dir = Path().resolve().parent
sys.path.append(str(base_dir / "src"))
from helpers.load_data import load_data
import warnings
warnings.filterwarnings("ignore")

In [2]:
dataset = "telco_customer_churn_clean.csv"
X_train, X_test, y_train, y_test = load_data(dataset)

Dataset 'telco_customer_churn_clean.csv' loaded successfully.

TotalCharges was typecasted to numerical.

Null values were removed.


Dataset size : 
7032 rows
21 columns


Successfully dropped the columns
   -customerID
   -gender
   -PhoneService
   -TotalCharges


Dataset split complete.


In [3]:
model1 = LogisticRegression()

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

metrics = ["accuracy", "precision", "recall", "f1"]

scores_model1 = cross_validate(
    estimator=model1,
    X=X_train,
    y=y_train,
    cv=skf,
    scoring=metrics
)
scores_model2 = {}
c = [0.01, 0.03, 0.09, 0.3, 0.9, 3]
for c_val in c:
    model2 = LogisticRegression(
        solver='liblinear',
        penalty='l2',
        C=c_val,
        l1_ratio=0
    )
    
    scores_model = cross_validate(
        estimator=model2,
        X=X_train,
        y=y_train,
        cv=skf,
        scoring=metrics
    )

    scores_model2[c_val] = scores_model

In [4]:
print("Model 1 scores\n\n")
for metric in metrics:
    mean_score = scores_model1[f"test_{metric}"].mean()
    print(f"Metric - {metric}\nMean Score - {mean_score}\n\n")

Model 1 scores


Metric - accuracy
Mean Score - 0.8


Metric - precision
Mean Score - 0.650933103919229


Metric - recall
Mean Score - 0.5351170568561873


Metric - f1
Mean Score - 0.5871049178249977




In [5]:
print("Model 2 scores\n\n")

for c_val in c:
    print(f"C value - {c_val}")
    curr = scores_model2[c_val]
    for metric in metrics:
        mean_score = curr[f"test_{metric}"].mean()
        print(f"Metric - {metric}\nMean Score - {mean_score}")
    print()
    print()

Model 2 scores


C value - 0.01
Metric - accuracy
Mean Score - 0.7950222222222222
Metric - precision
Mean Score - 0.6561864772425204
Metric - recall
Mean Score - 0.4809364548494983
Metric - f1
Mean Score - 0.5546608117501085


C value - 0.03
Metric - accuracy
Mean Score - 0.7964444444444444
Metric - precision
Mean Score - 0.6521057136637413
Metric - recall
Mean Score - 0.5023411371237458
Metric - f1
Mean Score - 0.5671184247259332


C value - 0.09
Metric - accuracy
Mean Score - 0.7971555555555556
Metric - precision
Mean Score - 0.6499243882775538
Metric - recall
Mean Score - 0.5150501672240803
Metric - f1
Mean Score - 0.5742153182603555


C value - 0.3
Metric - accuracy
Mean Score - 0.8010666666666667
Metric - precision
Mean Score - 0.6566898225991011
Metric - recall
Mean Score - 0.5284280936454849
Metric - f1
Mean Score - 0.5851192111026843


C value - 0.9
Metric - accuracy
Mean Score - 0.8005333333333334
Metric - precision
Mean Score - 0.6529010711783745
Metric - recall
Mean Score - 

In [6]:
df_dict = {}
for i,j in scores_model1.items():
    df_dict[i]=j.mean()
lr1_df = pd.DataFrame(data=df_dict.values(), index=df_dict.keys(), columns=["Mean_Score"])
lr1_df.to_csv(base_dir / "data" / "log_reg_no_regu.csv")

In [7]:
zeros = np.zeros((4,6))
lr2_df = pd.DataFrame(data=zeros,index=['test_accuracy','test_precision','test_recall','test_f1'],columns=c)
for c_val in c:
    curr = scores_model2[c_val]
    for metric in metrics:
        mean_score = curr[f"test_{metric}"].mean()
        lr2_df.loc[f"test_{metric}",c_val] = mean_score

lr2_df.to_csv(base_dir / "data" / "log_reg_l2_regu_diff_c.csv")